
# Thí nghiệm 02 — Scenario 4: Huấn luyện và đánh giá Train/Test

## 1. Thiết lập dữ liệu và môi trường

### 1.1. Mục tiêu

Notebook thực hiện huấn luyện và đánh giá bốn mô hình dự báo chuỗi thời gian trên dữ liệu tốc độ gió CMIP6, sử dụng phép chia Train/Test theo thời gian.

Các mô hình được khảo sát gồm:

1. LSTM.
2. PatchTST kết hợp toán tử Koopman.
3. Autoformer kết hợp toán tử Koopman.
4. Informer kết hợp toán tử Koopman.

Mỗi mô hình được huấn luyện trên tập Train và đánh giá trên tập Test chưa được sử dụng để cập nhật trọng số.

### 1.2. Nguồn dữ liệu

Dữ liệu được chuẩn bị trong:

`08_climate_train_test_data.ipynb`

Tệp đầu vào:

`data/scenario4_wind_train_test.pt`

Dữ liệu có 5 kênh tốc độ gió tại các vị trí:

`[0, 5000, 10000, 15000, 20000]`

Tập dữ liệu gốc gồm 2920 thời điểm, được chia thành:

| Tập | Số thời điểm | Số kênh |
|---|---:|---:|
| Train | 2336 | 5 |
| Test | 584 | 5 |

Dữ liệu được chia theo thứ tự thời gian, không xáo trộn.

### 1.3. Tiền xử lý

Notebook Data đã chuẩn hóa dữ liệu bằng `MinMaxScaler`.

Scaler được khớp trên tập Train và áp dụng cho tập Test.

Notebook Training đọc trực tiếp:

- `train_scaled`.
- `test_scaled`.

Không thực hiện lại bước chuẩn hóa để tránh tạo ra dữ liệu khác với lần thực nghiệm đã lưu.

### 1.4. Cửa sổ dự báo

Độ dài chuỗi đầu vào được cố định:

$$
L=120
$$

Với mỗi chân trời dự báo $H$, các cửa sổ được tạo riêng trên Train và Test.

Input:

$$
X\in\mathbb{R}^{N\times120\times5}
$$

Target:

$$
Y\in\mathbb{R}^{N\times5H}
$$

Công thức tạo cửa sổ được giữ nguyên:

`range(len(data) - seq_len - horizon)`

### 1.5. Không gian Grid Search

Patch lengths:

$$
P\in\{40,50,60,70,80,90,100\}
$$

Forecast horizons:

$$
H\in\{2,4,6,8,10,12,14,16\}
$$

Tổng số lượt huấn luyện:

$$
7\times8\times4=224
$$

Cấu hình chung:

| Tham số | Giá trị |
|---|---|
| seq_len | 120 |
| d_model | 48 |
| epochs | 4000 |
| learning_rate | 0.0003 |
| lyap_weight | 0.1 |
| seed | 7 |

### 1.6. Quản lý kết quả

Các checkpoint được lưu tại:

`results/scenario4_train_test_e4000/`

Mỗi checkpoint lưu trọng số cuối cùng, dự báo, nhãn, metric của cả Train và Test, lịch sử huấn luyện và thông tin phục vụ việc tiếp tục thực nghiệm.

Tệp `metrics.csv` chứa hai dòng cho mỗi cấu hình: một dòng Train và một dòng Test.

### 1.7. Bảo toàn thực nghiệm

Thực nghiệm đã tạo đầy đủ 224 checkpoint.

Trong lần tái cấu trúc repository:

- Không huấn luyện lại mô hình.
- Không thay đổi trọng số.
- Không tạo lại dữ liệu.
- Không thay đổi cách tạo cửa sổ.
- Không ghi đè checkpoint hoặc metrics.
- Không tạo thêm bản ZIP kết quả không cần thiết.

Notebook giữ mã nguồn huấn luyện để phục vụ khả năng tái lập và kiểm toán về sau.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# 1. Xác định môi trường thực thi
kaggle_input = Path("/kaggle/input")

if kaggle_input.is_dir():
    # Tìm dữ liệu và model trong Kaggle Datasets
    data_files = list(
        kaggle_input.rglob("scenario4_wind_train_test.pt")
    )

    model_files = list(
        kaggle_input.rglob("02_climate_model.ipynb")
    )

    if len(data_files) != 1:
        raise FileNotFoundError(
            f"Expected one Scenario 4 dataset: {data_files}"
        )

    if len(model_files) != 1:
        raise FileNotFoundError(
            f"Expected one model notebook: {model_files}"
        )

    data_path = data_files[0]
    model_notebook = model_files[0]

    experiment_dir = Path("/kaggle/working")
    results_dir = (
        experiment_dir / "scenario4_train_test_e4000"
    )

else:
    # Tìm project root khi sử dụng máy local
    project_root = next(
        (
            p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "experiments" / "02_climate").is_dir()
        ),
        None
    )

    if project_root is None:
        raise FileNotFoundError(
            "Không tìm thấy thư mục gốc của project."
        )

    experiment_dir = (
        project_root / "experiments" / "02_climate"
    )

    data_path = (
        experiment_dir / "data"
        / "scenario4_wind_train_test.pt"
    )

    model_notebook = (
        experiment_dir / "02_climate_model.ipynb"
    )

    results_dir = (
        experiment_dir
        / "results"
        / "scenario4_train_test_e4000"
    )

    if not data_path.is_file():
        raise FileNotFoundError(data_path)

    if not model_notebook.is_file():
        raise FileNotFoundError(model_notebook)


# 2. Nạp dữ liệu đã được chuẩn bị ở file 08
dataset = torch.load(
    data_path,
    map_location="cpu",
    weights_only=True
)

train_scaled = dataset["train_scaled"].numpy()
test_scaled = dataset["test_scaled"].numpy()

location_indices = dataset["location_indices"]


# 3. Nạp cấu hình thực nghiệm từ dataset
seq_len = dataset["seq_len"]
patch_lengths = dataset["patch_lengths"]
horizons = dataset["horizons"]

d_model = dataset["d_model"]
epochs = dataset["epochs"]
learning_rate = dataset["learning_rate"]
lyap_weight = dataset["lyap_weight"]
seed = dataset["seed"]

input_dim = train_scaled.shape[1]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 4. Hàm xây dựng cửa sổ trượt
def build_windows(data, seq_len, horizon):
    X, Y = [], []

    for i in range(len(data) - seq_len - horizon):
        X.append(data[i:i + seq_len])

        Y.append(
            data[i + seq_len:i + seq_len + horizon]
            .T.reshape(-1)
        )

    x = torch.tensor(
        np.stack(X), dtype=torch.float32
    )

    y = torch.tensor(
        np.stack(Y), dtype=torch.float32
    )

    return x, y


# 5. Kiểm tra kích thước với Horizon = 10
x_train, y_train = build_windows(
    train_scaled, seq_len, 10
)

x_test, y_test = build_windows(
    test_scaled, seq_len, 10
)


# 6. Xác minh dữ liệu và cấu hình
assert train_scaled.shape == (2336, 5)
assert test_scaled.shape == (584, 5)

assert np.isfinite(train_scaled).all()
assert np.isfinite(test_scaled).all()

assert seq_len == 120
assert patch_lengths == [40, 50, 60, 70, 80, 90, 100]
assert horizons == [2, 4, 6, 8, 10, 12, 14, 16]
assert d_model == 48
assert epochs == 4000

assert x_train.shape == (2206, 120, 5)
assert y_train.shape == (2206, 50)

assert x_test.shape == (454, 120, 5)
assert y_test.shape == (454, 50)


# 7. Tổng hợp dữ liệu
print("=== SCENARIO 4 — TRAIN/TEST ===")

print("Device:", device)
print("Dataset:", data_path)
print("Model notebook:", model_notebook)

print("\nData:")
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)
print("Locations:", location_indices)

print("\nSliding windows (H=10):")
print("X_train:", x_train.shape)
print("Y_train:", y_train.shape)
print("X_test:", x_test.shape)
print("Y_test:", y_test.shape)

print("\nExperiment:")
print("Sequence length:", seq_len)
print("Patch lengths:", patch_lengths)
print("Horizons:", horizons)
print("d_model:", d_model)
print("Epochs:", epochs)

print(
    "Total training runs:",
    len(patch_lengths) * len(horizons) * 4
)

print("\nResults:", results_dir)

=== SCENARIO 4 - KAGGLE ===
GPU: Tesla T4
Dataset: /kaggle/input/datasets/tunnghanguynng/scenario4/scenario4_wind_train_test.pt
Model notebook: /kaggle/input/datasets/tunnghanguynng/climate/02_climate_model.ipynb

Data:
Train: (2336, 5)
Test: (584, 5)
Locations: [0, 5000, 10000, 15000, 20000]

Sliding windows (H=10):
X_train: torch.Size([2206, 120, 5])
Y_train: torch.Size([2206, 50])
X_test: torch.Size([454, 120, 5])
Y_test: torch.Size([454, 50])

Experiment:
Sequence length: 120
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]
d_model: 48
Epochs: 4000
Total training runs: 224

Results: /kaggle/working/scenario4_train_test_e4000



## 2. Nạp kiến trúc và khởi tạo mô hình

### 2.1. Mục tiêu

Notebook sử dụng các kiến trúc đã được định nghĩa trong:

`02_climate_model.ipynb`

Các lớp được nạp từ mã nguồn hiện có, không xây dựng lại kiến trúc riêng cho Scenario 4.

### 2.2. Trích xuất định nghĩa mô hình

Notebook Model được đọc dưới dạng JSON.

Thư viện `ast` được sử dụng để trích xuất:

- Import.
- Định nghĩa hàm.
- Định nghĩa lớp.

Các định nghĩa sau đó được biên dịch và đưa vào môi trường Python hiện tại.

Cơ chế này tránh thực thi toàn bộ notebook Model khi chỉ cần sử dụng kiến trúc.

### 2.3. Model Factory

Hàm `create_model(model_name, patch_len, horizon)` khởi tạo một trong bốn mô hình:

| Tên | Lớp |
|---|---|
| LSTM | `SimpleLSTMForecaster` |
| PatchTST | `Koopformer_PatchTST` |
| Autoformer | `Koopformer_Autoformer` |
| Informer | `Koopformer_Informer` |

Các mô hình nhận 5 kênh đầu vào.

LSTM sử dụng hidden size 48 và 2 tầng.

Các backbone còn lại sử dụng cấu hình `d_model=48`.

### 2.4. Phân biệt seq_len và patch_len

Độ dài chuỗi đầu vào được cố định:

$$
L=120
$$

Trong khi đó:

$$
P\in\{40,50,60,70,80,90,100\}
$$

được truyền vào các mô hình dùng patch.

Vì vậy, việc thay đổi patch length không làm thay đổi số thời điểm có trong tensor đầu vào.

### 2.5. Bảo toàn checkpoint

Cấu trúc và tham số của các lớp mô hình phải được giữ nguyên để tương thích với những checkpoint đã lưu.

Không thay đổi tên lớp, các tham số khởi tạo hoặc phương thức `forward()` trong lần tái cấu trúc này.


In [4]:
import ast
import json
import torch

# 1. Extract model definitions without running local dataset code
with open(model_notebook, "r", encoding="utf-8") as f:
    notebook = json.load(f)

allowed_nodes = (
    ast.Import,
    ast.ImportFrom,
    ast.FunctionDef,
    ast.ClassDef
)

definitions = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = cell["source"]
    if isinstance(source, list):
        source = "".join(source)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, allowed_nodes):
            definitions.append(node)

module = ast.Module(body=definitions, type_ignores=[])
ast.fix_missing_locations(module)

exec(
    compile(module, str(model_notebook), "exec"),
    globals()
)

# 2. Model factory: seq_len fixed, patch_len variable
model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]

def create_model(model_name, patch_len, horizon):
    if model_name == "LSTM":
        return SimpleLSTMForecaster(
            input_dim=5,
            hidden_dim=48,
            num_layers=2,
            horizon=horizon
        )

    if model_name == "PatchTST":
        return Koopformer_PatchTST(
            input_dim, seq_len, horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    if model_name == "Autoformer":
        return Koopformer_Autoformer(
            input_dim, seq_len, horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    if model_name == "Informer":
        return Koopformer_Informer(
            input_dim, seq_len, horizon,
            patch_len=patch_len,
            d_model=d_model
        )

    raise ValueError(f"Unknown model: {model_name}")


=== SCENARIO 4 - MODEL VALIDATION ===

Patch length=40, Horizon=8
LSTM         Output=(2, 40) Params=31,336
PatchTST     Output=(2, 40) Params=73,192
Autoformer   Output=(2, 40) Params=53,552
Informer     Output=(2, 40) Params=73,144

Patch length=70, Horizon=10
LSTM         Output=(2, 50) Params=31,826
PatchTST     Output=(2, 50) Params=80,882
Autoformer   Output=(2, 50) Params=55,116
Informer     Output=(2, 50) Params=80,834

Patch length=100, Horizon=16
LSTM         Output=(2, 80) Params=33,296
PatchTST     Output=(2, 80) Params=89,552
Autoformer   Output=(2, 80) Params=60,144
Informer     Output=(2, 80) Params=89,504

All model configurations validated.



## 3. Hàm huấn luyện và đánh giá mô hình

### 3.1. Mục tiêu

Notebook định nghĩa hai hàm chính:

- `train_scenario4()`: huấn luyện mô hình trên tập Train.
- `evaluate_scenario4()`: đánh giá mô hình trên tập Test.

Việc tách hai hàm giúp bảo đảm rằng Test chỉ được sử dụng để tính kết quả dự báo, không tham gia cập nhật trọng số.

### 3.2. Hàm mất mát dự báo

Hàm mất mát chính là Mean Squared Error:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{N}
\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

Trong đó:

- $\hat y_i$: giá trị dự báo.
- $y_i$: giá trị thực tế.
- $N$: số phần tử trong tensor đầu ra.

### 3.3. Chính quy hóa Lyapunov

Các biến thể Koopman sử dụng thêm thành phần chính quy hóa nhằm hạn chế sự gia tăng năng lượng trong không gian tiềm ẩn.

Gọi $z$ và $z'$ là biểu diễn trước và sau toán tử Koopman.

Ma trận Lyapunov:

$$
P=I
$$

Hàm năng lượng:

$$
V(z)=z^\top Pz
$$

Thành phần phạt:

$$
\mathcal{L}_{\mathrm{Lyap}}
=
\frac{1}{B}
\sum_{i=1}^{B}
\max(0,V(z'_i)-V(z_i))
$$

Trong đó $B$ là số mẫu trong batch.

### 3.4. Hàm mục tiêu tổng hợp

$$
\mathcal{L}
=
\mathcal{L}_{\mathrm{MSE}}
+
\lambda\mathcal{L}_{\mathrm{Lyap}}
$$

Giá trị hệ số:

| Mô hình | $\lambda$ |
|---|---:|
| LSTM | 0 |
| PatchTST + Koopman | 0.1 |
| Autoformer + Koopman | 0.1 |
| Informer + Koopman | 0.1 |

LSTM được sử dụng làm mô hình đối chứng và không áp dụng chính quy hóa Lyapunov.

### 3.5. Huấn luyện trên tập Train

Hàm `train_scenario4()` thực hiện:

1. Chuyển mô hình và các cửa sổ Train lên thiết bị tính toán.
2. Khởi tạo Adam Optimizer.
3. Khởi tạo ma trận Lyapunov $P=I$.
4. Thực hiện truyền xuôi trên toàn bộ cửa sổ Train.
5. Tính MSE và thành phần chính quy hóa Lyapunov.
6. Lan truyền ngược gradient.
7. Cập nhật trọng số mô hình.
8. Ghi nhận lịch sử hàm mất mát và chỉ số theo dõi Koopman.

Sau 4000 epoch, mô hình chuyển sang chế độ `eval()` để tạo dự báo trên Train.

### 3.6. Đánh giá trên tập Test

Hàm `evaluate_scenario4()` sử dụng:

`model.eval()`

và:

`torch.no_grad()`

để tạo dự báo trên các cửa sổ Test.

Không gọi:

- `loss.backward()`.
- `optimizer.step()`.

Do đó, quá trình đánh giá Test không cập nhật trọng số mô hình.

### 3.7. Chỉ số đánh giá

Hai chỉ số được sử dụng:

**Mean Squared Error:**

$$
\mathrm{MSE}
=
\frac{1}{N}\sum_{i=1}^{N}
(\hat y_i-y_i)^2
$$

**Mean Absolute Error:**

$$
\mathrm{MAE}
=
\frac{1}{N}\sum_{i=1}^{N}
|\hat y_i-y_i|
$$

Cả MSE và MAE đều được tính riêng cho Train và Test.

Giá trị sai số được tính trong không gian dữ liệu đã chuẩn hóa Min-Max.

### 3.8. Các giá trị được ghi nhận

Hàm huấn luyện trả về:

| Giá trị | Nội dung |
|---|---|
| `pred_train` | Dự báo trên Train |
| `losses` | Lịch sử hàm mất mát theo epoch |
| `eigs` | Lịch sử chỉ số theo dõi tham số Koopman |

Hàm đánh giá trả về:

| Giá trị | Nội dung |
|---|---|
| `pred_test` | Dự báo trên Test |
| `test_mse` | MSE trên Test |
| `test_mae` | MAE trên Test |

Trong code, `eigs` được tính từ `max(abs(sigmoid(S_raw)))`.

Đại lượng này chưa nhân hệ số giới hạn 0.99 và không phải phép tính trực tiếp trị riêng lớn nhất của toán tử Koopman.

### 3.9. Giới hạn đánh giá

Scenario 4 có tập Test riêng, cho phép đánh giá mô hình trên khoảng thời gian không được sử dụng để huấn luyện.

Tuy nhiên, nếu lựa chọn cấu hình tốt nhất dựa trên chính Test, kết quả của cấu hình được lựa chọn không còn là đánh giá hoàn toàn độc lập đối với quá trình lựa chọn mô hình.

Notebook hiện không xây dựng một tập Validation riêng.


In [5]:
import torch.nn as nn
import torch.optim as optim

# 1. Thiết lập hạt giống ngẫu nhiên
def set_seed(seed=7):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

# 2. Truy cập các thuộc tính lồng nhau của mô hình
def _nested_getattr(obj, path):
    for p in path.split("."):
        obj = getattr(obj, p, None)
        if obj is None:
            return None
    return obj

# 3. Huấn luyện chỉ trên các cửa sổ Train
def train_scenario4(
    model, x_train, y_train,
    epochs=4000, lr=3e-4,
    koop_attr="koop", lyap_weight=0.1
):
    model.to(device)
    x_train = x_train.to(device)
    y_train = y_train.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    losses, eigs = [], []

    # Khởi tạo ma trận Lyapunov P = I
    with torch.no_grad():
        try:
            _, z, z_next = model(
                x_train[:1], return_latents=True
            )
        except ValueError:
            z = z_next = torch.zeros(
                1, model.fc.in_features, device=device
            )

    P = torch.eye(z.shape[1], device=device)

    for ep in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred, z, z_next = model(
            x_train, return_latents=True
        )

        pred_loss = mse(pred, y_train)

        if lyap_weight:
            zp = torch.einsum("bi,ij->bj", z, P)
            znp = torch.einsum("bi,ij->bj", z_next, P)

            lyap = torch.relu(
                (znp * z_next).sum(1)
                - (zp * z).sum(1)
            ).mean()
        else:
            lyap = torch.zeros(1, device=device)

        loss = pred_loss + lyap_weight * lyap

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Theo dõi tham số giá trị kỳ dị Koopman
        obj = (
            _nested_getattr(model, koop_attr)
            if koop_attr else None
        )

        eig_raw = (
            obj.S_raw
            if isinstance(obj, StrictStableKoopmanOperator)
            else None
        )

        eigs.append(
            0.0 if eig_raw is None
            else torch.sigmoid(eig_raw).abs().max().item()
        )

        if ep % max(1, epochs // 10) == 0 or ep == epochs - 1:
            print(
                f"Epoch {ep:>4}/{epochs} "
                f"MSE {pred_loss.item():.5e} "
                f"Lyap {(lyap_weight * lyap).item():.5e}"
            )

    # Dự báo trên các cửa sổ Train
    model.eval()

    with torch.no_grad():
        pred_train = model(x_train).cpu().numpy()

    return (
        pred_train,
        np.asarray(losses),
        np.asarray(eigs)
    )

# 4. Đánh giá trên các cửa sổ Test
def evaluate_scenario4(model, x_test, y_test):
    model.to(device)
    model.eval()

    with torch.no_grad():
        pred_test = model(x_test.to(device)).cpu().numpy()

    truth_test = y_test.cpu().numpy()

    error = pred_test - truth_test

    mse_value = float(np.mean(error ** 2))
    mae_value = float(np.mean(np.abs(error)))

    return pred_test, mse_value, mae_value

print("=== SCENARIO 4 FUNCTIONS READY ===")
print("Device:", device)
print("Epochs:", epochs)
print("Learning rate:", learning_rate)
print("Lyapunov weight:", lyap_weight)
print("Models:", model_names)


=== SCENARIO 4 FUNCTIONS READY ===
Device: cuda
Epochs: 4000
Learning rate: 0.0003
Lyapunov weight: 0.1
Models: ['LSTM', 'PatchTST', 'Autoformer', 'Informer']



## 4. Grid Search và lưu checkpoint Scenario 4

### 4.1. Mục tiêu

Notebook thực hiện Grid Search trên 224 cấu hình dự báo tốc độ gió CMIP6.

Mỗi cấu hình được xác định bởi:

$$
(P,H,M)
$$

Trong đó:

- $P$: patch length.
- $H$: forecast horizon.
- $M$: kiến trúc mô hình.

Các cấu hình được duyệt theo thứ tự trong `itertools.product()`.

### 4.2. Không gian thực nghiệm

Patch lengths:

$$
P\in\{40,50,60,70,80,90,100\}
$$

Forecast horizons:

$$
H\in\{2,4,6,8,10,12,14,16\}
$$

Bốn mô hình:

- LSTM.
- PatchTST + Koopman.
- Autoformer + Koopman.
- Informer + Koopman.

Tổng số lượt huấn luyện:

$$
7\times8\times4=224
$$

Mỗi lượt gồm 4000 epoch.

### 4.3. Quy trình cho mỗi cấu hình

1. Tạo cửa sổ Train và Test theo forecast horizon.
2. Khởi tạo mô hình với patch length tương ứng.
3. Huấn luyện mô hình chỉ trên Train.
4. Tạo dự báo Train.
5. Đánh giá trên Test mà không cập nhật trọng số.
6. Tính MSE và MAE cho hai tập.
7. Lưu checkpoint, lịch sử huấn luyện và metric.

### 4.4. Định danh dữ liệu

Notebook sử dụng SHA-256 để tạo fingerprint từ các mảng `train_scaled` và `test_scaled`.

Fingerprint được tính từ biểu diễn byte của hai mảng theo thứ tự:

`train_scaled.tobytes() + test_scaled.tobytes()`

Giá trị này được lưu trong trường `data_hash`.

Khi đọc lại checkpoint, notebook so sánh `data_hash` với dữ liệu đang được nạp.

Cơ chế này giúp phát hiện checkpoint được tạo từ một bộ dữ liệu khác.

### 4.5. Cơ chế checkpoint

Mỗi mô hình được lưu với tên:

`{model}_p{patch_length}_h{horizon}_d48.pt`

Ví dụ:

`patchtst_p40_h8_d48.pt`

Checkpoint chứa:

| Nhóm | Nội dung |
|---|---|
| Mô hình | `model`, `model_state_dict` |
| Train | `pred_train`, `truth_train`, `train_mse`, `train_mae` |
| Test | `pred_test`, `truth_test`, `test_mse`, `test_mae` |
| Lịch sử | `loss`, `eig` |
| Cấu hình | `patch_len`, `horizon`, `seq_len`, `d_model`, `input_dim`, `epochs` |
| Kiểm tra | `data_hash` |
| RNG | `rng_torch`, `rng_cuda` |

### 4.6. Tiếp tục thực nghiệm

Trước khi huấn luyện một cấu hình, code kiểm tra checkpoint tương ứng.

Nếu checkpoint tồn tại và metadata hợp lệ, kết quả được tái sử dụng.

Nếu checkpoint chưa tồn tại, quá trình huấn luyện chỉ được bắt đầu khi người thực hiện chủ động cho phép.

Trạng thái ngẫu nhiên được lưu và khôi phục để hỗ trợ tiếp tục các lượt huấn luyện tuần tự.

### 4.7. Quản lý metrics

Tệp:

`results/scenario4_train_test_e4000/metrics.csv`

Bao gồm các cột:

| Cột | Ý nghĩa |
|---|---|
| `Model` | Tên mô hình |
| `PatchLen` | Patch length |
| `Horizon` | Forecast horizon |
| `d_model` | Kích thước biểu diễn |
| `Set` | Train hoặc Test |
| `Epochs` | Số epoch |
| `MSE` | Sai số bình phương trung bình |
| `MAE` | Sai số tuyệt đối trung bình |

Mỗi cấu hình có hai dòng kết quả.

Với 224 cấu hình, tổng số dòng kỳ vọng là:

$$
224\times2=448
$$

### 4.8. Kết quả hiện có

Thư mục kết quả trên GitHub đã chứa:

- 224 checkpoint.
- 448 dòng metrics.
- 224 kết quả Train.
- 224 kết quả Test.

Các kết quả này được bảo toàn và không cần huấn luyện lại.

### 4.9. Nguyên tắc vận hành an toàn

Notebook được tổ chức theo hai chế độ:

- **Kiểm tra:** chỉ đọc dữ liệu, checkpoint và metrics; không huấn luyện hoặc ghi kết quả.
- **Huấn luyện:** thực hiện các cấu hình chưa hoàn thành khi được chủ động bật.

Chế độ mặc định là kiểm tra.

Không tự động tạo bản ZIP chứa toàn bộ checkpoint, vì những tệp kết quả đã được lưu trong repository và bản ZIP bổ sung sẽ làm tăng dung lượng ổ đĩa.


In [6]:

import gc
import hashlib
import itertools
import shutil


# 1. Chế độ vận hành
# False: chỉ kiểm tra kết quả đã lưu, không huấn luyện.
# True: cho phép huấn luyện những cấu hình chưa có checkpoint.
RUN_TRAINING = False

# Số cấu hình mới tối đa trong một lần huấn luyện
RUNS_PER_BATCH = 224

# Chỉ dùng khi chủ động tiếp tục huấn luyện trên Kaggle
RESTORE_KAGGLE_CHECKPOINTS = False


# 2. Cấu hình Grid Search
model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]

configs = list(itertools.product(
    patch_lengths,
    horizons,
    model_names
))

assert len(configs) == 224
assert seq_len == 120
assert d_model == 48
assert epochs == 4000
assert input_dim == 5

metrics_path = results_dir / "metrics.csv"


# 3. Fingerprint của bộ dữ liệu Scenario 4
data_hash = hashlib.sha256(
    train_scaled.tobytes() + test_scaled.tobytes()
).hexdigest()


# 4. Xác định đường dẫn checkpoint
def checkpoint_path(p, h, name):
    filename = (
        f"{name.lower()}_p{p}_h{h}_d{d_model}.pt"
    )
    return results_dir / filename


# 5. Kiểm tra metadata checkpoint
def validate_checkpoint(ckpt, p, h, name):
    assert ckpt["model"] == name
    assert ckpt["patch_len"] == p
    assert ckpt["horizon"] == h

    assert ckpt["seq_len"] == seq_len
    assert ckpt["d_model"] == d_model
    assert ckpt["input_dim"] == input_dim
    assert ckpt["epochs"] == epochs

    assert ckpt["data_hash"] == data_hash

    for key in (
        "model_state_dict",
        "pred_train",
        "truth_train",
        "pred_test",
        "truth_test",
        "train_mse",
        "train_mae",
        "test_mse",
        "test_mae",
    ):
        assert key in ckpt, f"Missing key: {key}"


# 6. Tạo một dòng kết quả Train hoặc Test
def make_record(name, p, h, set_name, mse, mae):
    return {
        "Model": name,
        "PatchLen": p,
        "Horizon": h,
        "d_model": d_model,
        "Set": set_name,
        "Epochs": epochs,
        "MSE": float(mse),
        "MAE": float(mae),
    }


# 7. Tổng hợp metrics từ các checkpoint hiện có
# Chỉ được gọi khi RUN_TRAINING=True và có lượt train mới.
def update_metrics_from_checkpoints():
    rows = []

    for p, h, name in configs:
        path = checkpoint_path(p, h, name)

        if not path.is_file():
            continue

        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        validate_checkpoint(ckpt, p, h, name)

        rows.append(make_record(
            name, p, h, "Train",
            ckpt["train_mse"],
            ckpt["train_mae"]
        ))

        rows.append(make_record(
            name, p, h, "Test",
            ckpt["test_mse"],
            ckpt["test_mae"]
        ))

    df = pd.DataFrame(
        rows,
        columns=[
            "Model", "PatchLen", "Horizon",
            "d_model", "Set", "Epochs",
            "MSE", "MAE"
        ]
    )

    temp_path = metrics_path.with_suffix(".csv.tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(metrics_path)

    return len(rows) // 2


# 8. Chuẩn bị thư mục kết quả
if RUN_TRAINING:
    if device.type != "cuda":
        raise RuntimeError(
            "Huấn luyện Scenario 4 yêu cầu GPU CUDA."
        )

    results_dir.mkdir(parents=True, exist_ok=True)

    # Tùy chọn: khôi phục checkpoint từ Kaggle Dataset
    if RESTORE_KAGGLE_CHECKPOINTS:
        resume_dirs = [
            p for p in Path("/kaggle/input").rglob(
                "scenario4_train_test_e4000"
            )
            if p.is_dir()
        ]

        if len(resume_dirs) > 1:
            raise RuntimeError(
                "Có nhiều nguồn checkpoint Kaggle."
            )

        if resume_dirs:
            for source in resume_dirs[0].glob(
                "*_p*_h*_d*.pt"
            ):
                destination = results_dir / source.name

                if not destination.exists():
                    shutil.copy2(source, destination)

else:
    if not results_dir.is_dir():
        raise FileNotFoundError(results_dir)


# 9. Xác minh các checkpoint đã tồn tại
existing = []
missing = []

for p, h, name in configs:
    path = checkpoint_path(p, h, name)

    if not path.is_file():
        missing.append((p, h, name))
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    validate_checkpoint(ckpt, p, h, name)
    existing.append((p, h, name))

    del ckpt


print("=== SCENARIO 4 CHECKPOINT AUDIT ===")
print("Total configurations:", len(configs))
print("Existing checkpoints:", len(existing))
print("Missing checkpoints:", len(missing))
print("Results directory:", results_dir)
print("Dataset hash:", data_hash)


# 10. Kiểm tra bảng metrics hiện có mà không ghi file
if metrics_path.is_file():
    metrics_df = pd.read_csv(metrics_path)

    print("\nMetrics rows:", len(metrics_df))
    print(
        "Train rows:",
        (metrics_df["Set"] == "Train").sum()
    )
    print(
        "Test rows:",
        (metrics_df["Set"] == "Test").sum()
    )

    if len(existing) == 224:
        assert len(metrics_df) == 448
        assert (metrics_df["Set"] == "Train").sum() == 224
        assert (metrics_df["Set"] == "Test").sum() == 224

else:
    print("\nMetrics file not found:", metrics_path)


# 11. Chỉ huấn luyện khi được chủ động bật
if not RUN_TRAINING:
    print("\nREAD-ONLY MODE")
    print("Không huấn luyện.")
    print("Không ghi checkpoint.")
    print("Không ghi metrics.csv.")
    print("Không tạo ZIP.")

else:
    # Khôi phục hạt giống ngẫu nhiên
    set_seed(seed)

    new_runs = 0

    for run_idx, (p, h, name) in enumerate(
        configs, start=1
    ):
        path = checkpoint_path(p, h, name)

        # Bỏ qua checkpoint đã có và khôi phục RNG
        if path.is_file():
            ckpt = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

            validate_checkpoint(ckpt, p, h, name)

            if "rng_torch" in ckpt:
                torch.set_rng_state(ckpt["rng_torch"])

            if ckpt.get("rng_cuda"):
                torch.cuda.set_rng_state_all(
                    ckpt["rng_cuda"]
                )

            print(f"SKIP [{run_idx}/224] {path.name}")
            del ckpt
            continue

        # Giới hạn số cấu hình mới của lần chạy
        if new_runs >= RUNS_PER_BATCH:
            break

        print(
            f"\n[{run_idx}/224] "
            f"{name} | PatchLen={p}, Horizon={h}"
        )

        # Tạo cửa sổ Train/Test cho horizon hiện tại
        x_train, y_train = build_windows(
            train_scaled, seq_len, h
        )

        x_test, y_test = build_windows(
            test_scaled, seq_len, h
        )

        # Khởi tạo mô hình
        model = create_model(name, p, h)

        is_lstm = name == "LSTM"

        # Huấn luyện trên Train
        pred_train, loss, eig = train_scenario4(
            model,
            x_train,
            y_train,
            epochs=epochs,
            lr=learning_rate,
            koop_attr=None if is_lstm else "koop",
            lyap_weight=0.0 if is_lstm else lyap_weight
        )

        # Tính metric trên Train
        truth_train = y_train.numpy()
        train_error = pred_train - truth_train

        train_mse = float(np.mean(train_error ** 2))
        train_mae = float(np.mean(np.abs(train_error)))

        # Đánh giá Test, không cập nhật trọng số
        pred_test, test_mse, test_mae = evaluate_scenario4(
            model, x_test, y_test
        )

        truth_test = y_test.numpy()

        print(
            f"Train: MSE={train_mse:.6f} "
            f"MAE={train_mae:.6f}"
        )

        print(
            f"Test: MSE={test_mse:.6f} "
            f"MAE={test_mae:.6f}"
        )

        # Đóng gói checkpoint
        checkpoint = {
            "model": name,
            "model_state_dict": model.cpu().state_dict(),

            "pred_train": torch.from_numpy(
                pred_train.copy()
            ),
            "truth_train": torch.from_numpy(
                truth_train.copy()
            ),

            "pred_test": torch.from_numpy(
                pred_test.copy()
            ),
            "truth_test": torch.from_numpy(
                truth_test.copy()
            ),

            "loss": torch.from_numpy(loss.copy()),
            "eig": torch.from_numpy(eig.copy()),

            "train_mse": train_mse,
            "train_mae": train_mae,
            "test_mse": test_mse,
            "test_mae": test_mae,

            "patch_len": p,
            "horizon": h,
            "seq_len": seq_len,
            "d_model": d_model,
            "input_dim": input_dim,
            "epochs": epochs,
            "data_hash": data_hash,

            "rng_torch": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all(),
        }

        # Lưu checkpoint bằng file tạm
        temp_path = path.with_suffix(".pt.tmp")

        torch.save(checkpoint, temp_path)
        temp_path.replace(path)

        # Chỉ cập nhật CSV sau một lượt train mới
        completed = update_metrics_from_checkpoints()

        print("Checkpoint saved:", path)
        print("Completed:", completed, "/ 224")

        new_runs += 1

        # Giải phóng bộ nhớ GPU
        del (
            model, x_train, y_train, x_test, y_test,
            pred_train, pred_test,
            truth_train, truth_test,
            train_error, loss, eig, checkpoint
        )

        gc.collect()
        torch.cuda.empty_cache()

    print("\n=== TRAINING BATCH COMPLETE ===")
    print("New runs:", new_runs)
    print("Results:", results_dir)
